In [ ]:
import re

# index_service.py
class IndexService:
    def __init__(self):
        self.documents = {}
        self.index = {}

    def add_document(self, doc_data):
        """Add a document to the index"""
        doc_id = str(len(self.documents) + 1)
        self.documents[doc_id] = {**doc_data, 'id': doc_id}

        # Create inverted index
        words = re.findall(r'\w+', doc_data['content'].lower())
        for word in words:
            if word not in self.index:
                self.index[word] = set()
            self.index[word].add(doc_id)

        return self.documents[doc_id]

    def get_document(self, doc_id):
        """Retrieve a document by ID"""
        return self.documents.get(doc_id)

    def search_word(self, word):
        """Find documents containing a word"""
        word = word.lower()
        return self.index.get(word, set())

    def search_logical(self, terms, operator='AND'):
        """Search using logical operators"""
        if not terms:
            return set()

        sets = [self.search_word(term) for term in terms]

        if operator.upper() == 'OR':
            return set().union(*sets)
        else: # Default to AND
            return set.intersection(*sets) if sets else set()

In [ ]:
class QueryService:
    def __init__(self, index_service):
        self.index_service = index_service
        self.queries = {}

    def create_query(self, query_data):
        """Create and execute a search query with logic support"""
        try:
            query_id = str(len(self.queries) + 1)
            search_terms = query_data.get('terms', [])
            operator = query_data.get('operator', 'AND') # Default operator

            # Use the new logical search from IndexService
            results = self.index_service.search_logical(search_terms, operator)

            query = {
                'id': query_id,
                'terms': search_terms,
                'operator': operator,
                'results': list(results),
                'timestamp': query_data.get('timestamp', 'now')
            }
            self.queries[query_id] = query
            return query

        except Exception as e:
            return {'error': str(e)}

In [ ]:
# result_service.py
class ResultService:
    def __init__(self, index_service, query_service):
        self.index_service = index_service
        self.query_service = query_service
        self.results = {}
    def format_results(self, query_id):
        """Format search results for display"""
        try:
            query = self.query_service.queries.get(query_id)
            if not query:
                return {'error': 'Query not found'}

            formatted_results = []
            for doc_id in query['results']:
                doc = self.index_service.get_document(doc_id)
                if doc:
                    formatted_results.append({
                        'doc_id': doc_id,
                        'title': doc['title'],
                        'snippet': doc['content'][:100] + '...'
                    })
            result_id = str(len(self.results) + 1)
            result = {
                'id': result_id,
                'query_id': query_id,
                'formatted_results': formatted_results,
                'count': len(formatted_results)
            }
            self.results[result_id] = result
            return result

        except Exception as e:
            return {'error': str(e)}

In [ ]:
# ranking_service.py
class RankingService:
    def __init__(self, index_service):
        self.index_service = index_service

    def rank_results(self, doc_ids, search_terms):
        """Rank documents based on term frequency"""
        ranked_docs = []
        for doc_id in doc_ids:
            doc = self.index_service.get_document(doc_id)
            if not doc:
                continue

            content_lower = doc['content'].lower()
            score = 0
            for term in search_terms:
                score += content_lower.count(term.lower())

            ranked_docs.append({
                'doc_id': doc_id,
                'score': score,
                'title': doc['title'],
                'content': doc['content']
            })

        # Sort by score in descending order
        return sorted(ranked_docs, key=lambda x: x['score'], reverse=True)

In [ ]:
# Modified ResultService to use RankingService
class ResultServiceWithRanking(ResultService):
    def __init__(self, index_service, query_service, ranking_service):
        super().__init__(index_service, query_service)
        self.ranking_service = ranking_service

    def format_results(self, query_id):
        """Format and rank search results"""
        try:
            query = self.query_service.queries.get(query_id)
            if not query:
                return {'error': 'Query not found'}

            # Use the ranking service
            ranked_list = self.ranking_service.rank_results(query['results'], query['terms'])

            formatted_results = []
            for item in ranked_list:
                formatted_results.append({
                    'doc_id': item['doc_id'],
                    'score': item['score'],
                    'title': item['title'],
                    'snippet': item['content'][:100] + '...'
                })

            result_id = str(len(self.results) + 1)
            result = {
                'id': result_id,
                'query_id': query_id,
                'formatted_results': formatted_results,
                'count': len(formatted_results)
            }
            self.results[result_id] = result
            return result

        except Exception as e:
            return {'error': str(e)}

In [ ]:
def main_with_logical_operators():
    index_service = IndexService()
    query_service = QueryService(index_service)
    ranking_service = RankingService(index_service)
    result_service = ResultServiceWithRanking(index_service, query_service, ranking_service)

    # Setup data
    index_service.add_document({'title': 'Python Doc', 'content': 'Python is for cloud'})
    index_service.add_document({'title': 'Cloud Doc', 'content': 'Cloud services are scalable'})
    index_service.add_document({'title': 'Mixed Doc', 'content': 'Python cloud microservices'})

    print("--- Testing OR operator ---")
    q_or = query_service.create_query({'terms': ['python', 'scalable'], 'operator': 'OR'})
    res_or = result_service.format_results(q_or['id'])
    for r in res_or['formatted_results']: print(f"Found: {r['title']}")

    print("\n--- Testing AND operator ---")
    q_and = query_service.create_query({'terms': ['python', 'cloud'], 'operator': 'AND'})
    res_and = result_service.format_results(q_and['id'])
    for r in res_and['formatted_results']: print(f"Found: {r['title']}")

if __name__ == '__main__':
    main_with_logical_operators()

In [ ]:
def main():
    # Initialize services
    index_service = IndexService()
    query_service = QueryService(index_service)
    result_service = ResultService(index_service, query_service)

    # Add sample documents
    doc1 = index_service.add_document({
        'title': 'Python Programming',
        'content': 'Python is a popular programming language for cloud computing'
    })
    doc2 = index_service.add_document({
        'title': 'Cloud Services',
        'content': 'Cloud computing enables scalable microservices architecture'
    })
    print(f"Added documents: {doc1['id']}, {doc2['id']}")
    # Create and execute a search query
    query = query_service.create_query({
        'terms': ['cloud', 'computing']
    })
    print(f"Query results: {query}")

    # Format the results
    formatted_results = result_service.format_results(query['id'])
    print(f"Formatted results: {formatted_results}")

if __name__ == "__main__":
    main()
